# Noise Model Simulator - Iris (3 Klassen, 2 Features)

**Modelle:** Quantum Kernel SVM + VQC (RealAmplitudes)  
**Datensatz:** Iris (3 Klassen, 2 Features)  
**Backend:** AerSimulator + IBM Noise Model (ibm_kingston)  
**Referenz:** `02_iris_fair.ipynb` (Ideal Simulator)  

Simuliert Hardware-Rauschen lokal - kein QPU-Kontingent verbraucht.  
Zeigt den Effekt von Dekohärenz und Gate-Fehlern auf die Klassifikationsqualität.

## 0. Imports

In [14]:
import pandas as pd
df = pd.read_csv('Ergebnisse/ergebnisse.csv')
print(df[df['Backend'].str.contains('Noise', na=False)][['Modell', 'Datensatz', 'Backend']])

                         Modell                                     Datensatz  \
8    Quantum Kernel SVM (Noise)           Iris (3 Klassen, 2 Features, noise)   
9   VQC (RealAmplitudes, Noise)           Iris (3 Klassen, 2 Features, noise)   
10   Quantum Kernel SVM (Noise)  Breast Cancer (2 Klassen, 2 Features, noise)   
11  VQC (RealAmplitudes, Noise)  Breast Cancer (2 Klassen, 2 Features, noise)   

               Backend  
8   AerSimulator+Noise  
9   AerSimulator+Noise  
10  AerSimulator+Noise  
11  AerSimulator+Noise  


In [15]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.svm import SVC

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.optimizers import SPSA
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_aer.primitives import SamplerV2
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from utils import save_result

print("Imports OK")

Imports OK


## 1. Noise Model laden

In [16]:
# IBM Account verbinden
service = QiskitRuntimeService(
    channel='ibm_quantum_platform',
    token='v7vNLPiYw-WbnfQI6YGZZsO1aCts-YzPzIc3wj0gSqn5'
)

# Noise Model von ibm_kingston laden
backend = service.backend("ibm_kingston")
noise_model = NoiseModel.from_backend(backend)

# AerSimulator mit Noise Model
noisy_simulator = AerSimulator(noise_model=noise_model)

print(f"Noise Model geladen von: {backend.name}")
print(f"Basis-Gates: {noise_model.basis_gates}")

qiskit_runtime_service._discover_account:WARNING:2026-05-23 15:52:21,103: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-05-23 15:52:24,519: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-05-23 15:52:24,520: Using instance: open-instance, plan: open


Noise Model geladen von: ibm_kingston
Basis-Gates: ['cz', 'delay', 'id', 'if_else', 'measure', 'measure_2', 'reset', 'rz', 'sx', 'x']


## 2. Datensatz - identische Pipeline wie `02_iris_fair`

In [17]:
iris = load_iris()
X = iris.data[:, [0, 2]]
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train_sc.shape}  |  Test: {X_test_sc.shape}")

Train: (105, 2)  |  Test: (45, 2)


## 3. Quantum Kernel SVM (Noise Model)

In [18]:
feature_map_qk = zz_feature_map(feature_dimension=2, reps=2)

# Noisy Sampler + explizites Fidelity-Objekt
noisy_sampler = SamplerV2.from_backend(noisy_simulator)
pm = generate_preset_pass_manager(backend=noisy_simulator, optimization_level=1)
fidelity = ComputeUncompute(sampler=noisy_sampler, pass_manager=pm)
kernel = FidelityQuantumKernel(feature_map=feature_map_qk, fidelity=fidelity,
                                max_circuits_per_job=300)

start = time.time()
svc_q = SVC(kernel=kernel.evaluate)
svc_q.fit(X_train_sc, y_train)
train_time_qk = round(time.time() - start, 4)

start = time.time()
y_pred_qk = svc_q.predict(X_test_sc)
infer_time_qk = round(time.time() - start, 4)

acc_qk = accuracy_score(y_test, y_pred_qk)
f1_qk  = f1_score(y_test, y_pred_qk, average='weighted')

print(f"Accuracy: {acc_qk:.4f}  |  F1: {f1_qk:.4f}")
print(f"Training: {train_time_qk}s  |  Inferenz: {infer_time_qk}s")
print()
print(classification_report(y_test, y_pred_qk, target_names=iris.target_names))

Accuracy: 0.6889  |  F1: 0.6844
Training: 60.6197s  |  Inferenz: 51.4636s

              precision    recall  f1-score   support

      setosa       0.67      0.80      0.73        15
  versicolor       0.73      0.73      0.73        15
   virginica       0.67      0.53      0.59        15

    accuracy                           0.69        45
   macro avg       0.69      0.69      0.68        45
weighted avg       0.69      0.69      0.68        45



## 4. VQC (Noise Model)

In [ ]:
feature_map_vqc = zz_feature_map(feature_dimension=2, reps=1)
ansatz = real_amplitudes(num_qubits=2, reps=2)

# VQC mit Noisy AerSampler
noisy_sampler_vqc = SamplerV2.from_backend(noisy_simulator)

vqc = VQC(
    feature_map=feature_map_vqc,
    ansatz=ansatz,
    optimizer=SPSA(maxiter=150),
    sampler=noisy_sampler_vqc,
)

print("Starte VQC Training (Noise Model)...")
start = time.time()
vqc.fit(X_train_sc, y_train)
train_time_vqc = round(time.time() - start, 4)

start = time.time()
y_pred_vqc = vqc.predict(X_test_sc)
infer_time_vqc = round(time.time() - start, 4)

acc_vqc = accuracy_score(y_test, y_pred_vqc)
f1_vqc  = f1_score(y_test, y_pred_vqc, average='weighted')

print(f"Accuracy: {acc_vqc:.4f}  |  F1: {f1_vqc:.4f}")
print(f"Training: {train_time_vqc}s  |  Inferenz: {infer_time_vqc}s")
print()
print(classification_report(y_test, y_pred_vqc, target_names=iris.target_names))

No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


Starte VQC Training (Noise Model)...


## 5. Ergebnisse speichern

In [ ]:
DATENSATZ = "Iris (3 Klassen, 2 Features, noise)"

df = pd.read_csv('Ergebnisse/ergebnisse.csv')
df = df[~(df['Backend'].str.contains('Noise', na=False) & 
          df['Datensatz'].str.contains('Iris', na=False))]
df.to_csv('Ergebnisse/ergebnisse.csv', index=False)

save_result("Quantum Kernel SVM (Noise)",   DATENSATZ, "AerSimulator+Noise", acc_qk,  f1_qk,  train_time_qk,  infer_time_qk,  Feature_Map="zz_feature_map", Reps=2)
save_result("VQC (RealAmplitudes, Noise)",  DATENSATZ, "AerSimulator+Noise", acc_vqc, f1_vqc, train_time_vqc, infer_time_vqc, Feature_Map="zz_feature_map", Reps=1)

print("Gespeichert.")

## 6. Vergleich: Ideal vs. Noise Model

In [ ]:
print(df.dtypes)
print(df["Accuracy"].head(10))

In [ ]:
print(df["Accuracy"].unique())

In [ ]:
df = pd.read_csv("Ergebnisse/ergebnisse.csv")

df_ideal = df[df["Datensatz"] == "Iris (3 Klassen, 2 Features, fair)"][["Modell", "Accuracy", "F1"]].copy()
df_ideal.columns = ["Modell", "Accuracy (Ideal)", "F1 (Ideal)"]

df_noise = df[df["Datensatz"] == "Iris (3 Klassen, 2 Features, noise)"][["Modell", "Accuracy", "F1"]].copy()
df_noise.columns = ["Modell", "Accuracy (Noise)", "F1 (Noise)"]

# Modellnamen vereinheitlichen für Merge
df_noise["Modell"] = df_noise["Modell"].str.replace(" (Noise)", "", regex=False)

df_cmp = pd.merge(df_ideal, df_noise, on="Modell", how="outer")
df_cmp["Δ Accuracy"] = (df_cmp["Accuracy (Noise)"] - df_cmp["Accuracy (Ideal)"]).round(4)

print(df_cmp.to_string(index=False))